# Monte Carlo Simulation for Portfolio Risk and Option Pricing

**Author:** Arakya Kaushik · MSc Statistical Data Science, University College Dublin

This notebook uses Monte Carlo simulation for two problems that come up across banking and insurance:

1. **Portfolio risk.** Simulating a year of correlated returns for a portfolio of European insurance stocks and measuring tail risk with **Value at Risk (VaR)** and **Conditional VaR (Expected Shortfall)**, including the **99.5% one-year level used in Solvency II**.
2. **Option pricing.** Pricing a European call by risk-neutral simulation, checking it against the **Black-Scholes** closed form, and making the estimator more efficient with **antithetic variates** and **control variates**.

**Acknowledgement.** The starting point was the QuantPy tutorial *"Monte Carlo as a tool for Financial Math"*. I rewrote it and extended it with Black-Scholes validation, implied volatility, variance reduction, convergence analysis, a 99.5% Solvency II style VaR and a correlation/diversification study.

Run the cells top to bottom. Every random draw uses a fixed seed, so you get the same numbers each time.

## 0. Setup

In [ ]:
# If a package is missing, run once:  pip install -r requirements.txt
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import brentq

SEED = 42                              # fixed seed -> reproducible results
rng = np.random.default_rng(SEED)
FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": 0.3})
print("NumPy", np.__version__, "| pandas", pd.__version__)

## 1. What Monte Carlo simulation does

Many quantities in finance and insurance are **expectations**: an option's price, the average claim cost, the probability of a large loss. When the expectation has no neat formula, we can estimate it by:

1. simulating many random scenarios from a model,
2. computing the quantity of interest in each scenario,
3. averaging.

The **Law of Large Numbers** says the average converges to the true expectation. The **Central Limit Theorem** tells us how far off we are likely to be. With $M$ simulations, the estimate $\hat\mu$ has **standard error**

$$SE(\hat\mu) = \frac{\hat\sigma}{\sqrt{M}},$$

so a 95% confidence interval is $\hat\mu \pm 1.96\,SE$. Two consequences shape this whole notebook:

* Error shrinks like $1/\sqrt{M}$. **Halving the error costs 4x the simulations.**
* The other lever is to **reduce $\hat\sigma$** (variance reduction, section 6). That's often far cheaper than adding paths.

# Part A: Portfolio risk

## 2. Data: a portfolio of European insurers

Six large insurers and reinsurers, all listed in euro, so no currency conversion is needed. Daily adjusted closing prices come from Yahoo Finance over a fixed window, which keeps the results reproducible.

In [ ]:
TICKERS = {
    "ALV.DE": "Allianz",
    "CS.PA": "AXA",
    "MUV2.DE": "Munich Re",
    "G.MI": "Generali",
    "HNR1.DE": "Hannover Re",
    "AGS.BR": "Ageas",
}
START, END = "2023-01-01", "2025-12-31"

def load_prices(tickers, start, end):
    # Download adjusted close prices. Falls back to clearly labelled synthetic data offline.
    try:
        import yfinance as yf
        px = yf.download(list(tickers), start=start, end=end, auto_adjust=True, progress=False)["Close"]
        px = px.dropna(how="all").ffill().dropna()
        if px.empty:
            raise ValueError("empty download")
        return px.rename(columns=tickers), "Yahoo Finance"
    except Exception as e:
        print(f"Download failed ({e}). Using SYNTHETIC prices so the notebook still runs. "
              "Do not report these numbers.")
        n, k = 750, len(tickers)
        g = np.random.default_rng(0)
        A = g.normal(size=(k, k)); corr = A @ A.T; d = np.sqrt(np.diag(corr)); corr = corr / np.outer(d, d)
        corr = 0.5 * corr + 0.5 * np.full((k, k), 0.6); np.fill_diagonal(corr, 1)
        vols = g.uniform(0.012, 0.018, k)
        rets = g.multivariate_normal(np.full(k, 0.0004), corr * np.outer(vols, vols), size=n)
        idx = pd.bdate_range(start, periods=n)
        return pd.DataFrame(100 * np.cumprod(1 + rets, axis=0), index=idx, columns=list(tickers.values())), "SYNTHETIC"

prices, DATA_SOURCE = load_prices(TICKERS, START, END)
print("Source:", DATA_SOURCE, "|", prices.shape[0], "trading days,", prices.index[0].date(), "to", prices.index[-1].date())
prices.tail()

In [ ]:
returns = prices.pct_change().dropna()          # simple daily returns
mean_returns = returns.mean()
cov_matrix = returns.cov()

summary = pd.DataFrame({
    "Mean daily return %": 100 * mean_returns,
    "Daily volatility %": 100 * returns.std(),
    "Annualised volatility %": 100 * returns.std() * np.sqrt(252),
}).round(3)
display(summary)

plt.imshow(returns.corr(), cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="correlation")
plt.xticks(range(len(returns.columns)), returns.columns, rotation=45, ha="right")
plt.yticks(range(len(returns.columns)), returns.columns)
plt.title("Correlation of daily returns"); plt.grid(False); plt.tight_layout()
plt.savefig(FIG_DIR / "correlation_matrix.png", dpi=150); plt.show()

## 3. Simulating correlated returns with a Cholesky decomposition

Insurers' share prices move together, so we can't simulate each stock independently. We want random daily returns $R \sim N(\mu, \Sigma)$, where $\Sigma$ is the covariance matrix estimated above.

**Cholesky trick.** Factor $\Sigma = L L^\top$, where $L$ is lower triangular. If $Z \sim N(0, I)$ is a vector of *independent* standard normals, then

$$\mu + L Z \sim N(\mu, \Sigma),$$

because $\text{Cov}(LZ) = L\,I\,L^\top = \Sigma$. That one matrix multiply turns independent noise into correctly correlated returns.

Each simulated path compounds daily portfolio returns over one year (252 trading days):
$V_T = V_0 \prod_{t=1}^{252} (1 + w^\top R_t)$.

In [ ]:
def simulate_portfolio(mu, cov, weights, n_days, n_paths, rng, v0=10_000.0):
    # Return simulated portfolio values, shape (n_days + 1, n_paths).
    L = np.linalg.cholesky(cov)                              # cov = L @ L.T
    Z = rng.standard_normal((n_days, n_paths, len(weights)))  # independent N(0,1)
    asset_returns = mu + Z @ L.T                              # correlated returns
    port_returns = asset_returns @ weights
    values = v0 * np.cumprod(1 + port_returns, axis=0)
    return np.vstack([np.full(n_paths, v0), values])

V0 = 10_000.0
N_DAYS = 252          # one-year horizon, as in Solvency II
N_PATHS = 10_000
weights = np.full(len(mean_returns), 1 / len(mean_returns))   # equally weighted

t0 = time.perf_counter()
paths = simulate_portfolio(mean_returns.values, cov_matrix.values, weights, N_DAYS, N_PATHS, rng, V0)
print(f"Simulated {N_PATHS:,} paths x {N_DAYS} days in {time.perf_counter() - t0:.2f}s")

plt.plot(paths[:, :200], lw=0.6, alpha=0.5)
plt.xlabel("Trading day"); plt.ylabel("Portfolio value (€)")
plt.title("200 of the 10,000 simulated one-year paths"); plt.tight_layout()
plt.savefig(FIG_DIR / "portfolio_paths.png", dpi=150); plt.show()

## 4. Measuring tail risk: VaR and Expected Shortfall

Let the one-year **loss** be $\text{Loss} = V_0 - V_T$.

* **Value at Risk** at confidence $\alpha$ is the loss that's exceeded only with probability $1-\alpha$: the $\alpha$-quantile of the loss distribution.
* **Conditional VaR / Expected Shortfall (ES)** is the *average* loss in those worst $1-\alpha$ scenarios. It looks inside the tail, whereas VaR only marks where the tail starts, and it's a *coherent* risk measure (it rewards diversification).

**Why 99.5%?** Under **Solvency II**, the EU insurance capital regime, an insurer's Solvency Capital Requirement is calibrated to a **99.5% VaR over one year**, i.e. a 1-in-200-year loss.

In [ ]:
def var_es(final_values, v0, alpha):
    losses = v0 - final_values
    var = np.quantile(losses, alpha)
    es = losses[losses >= var].mean()
    return var, es

final = paths[-1]
rows = []
for a in (0.95, 0.99, 0.995):
    v, e = var_es(final, V0, a)
    rows.append({"Confidence": f"{a:.1%}", "VaR (€)": round(v, 0), "Expected Shortfall (€)": round(e, 0),
                 "VaR % of portfolio": round(100 * v / V0, 1)})
risk_table = pd.DataFrame(rows)
display(risk_table)

losses = V0 - final
v95, _ = var_es(final, V0, 0.95); v995, _ = var_es(final, V0, 0.995)
plt.hist(losses, bins=80, color="steelblue", alpha=0.8)
plt.axvline(v95, color="orange", lw=2, label=f"95% VaR = €{v95:,.0f}")
plt.axvline(v995, color="red", lw=2, label=f"99.5% VaR = €{v995:,.0f}")
plt.xlabel("One-year loss (€)  [negative = gain]"); plt.ylabel("Number of scenarios")
plt.title("Simulated one-year loss distribution"); plt.legend(); plt.tight_layout()
plt.savefig(FIG_DIR / "loss_distribution.png", dpi=150); plt.show()

### 4.1 How much does correlation matter?

We re-run the simulation with all correlations set to zero (same volatilities). If the stocks were independent, losses would partly cancel and the tail would look much smaller. **Ignoring correlation understates risk**, which is why capital models care so much about dependence.

In [ ]:
cov_independent = np.diag(np.diag(cov_matrix.values))       # keep variances, drop covariances
paths_ind = simulate_portfolio(mean_returns.values, cov_independent, weights, N_DAYS, N_PATHS,
                               np.random.default_rng(SEED), V0)
v_corr, e_corr = var_es(final, V0, 0.995)
v_ind, e_ind = var_es(paths_ind[-1], V0, 0.995)
print(f"99.5% VaR with real correlation : €{v_corr:,.0f}")
print(f"99.5% VaR assuming independence : €{v_ind:,.0f}")
print(f"-> Ignoring correlation understates 99.5% VaR by {100 * (1 - v_ind / v_corr):.0f}%")

### 4.2 Limitations of this risk model

* **Normal returns** have thin tails. Real returns show fat tails and volatility clustering, so the true 99.5% loss is probably larger. A Student-t or GARCH model would be the next step.
* **Constant mean and covariance** estimated from three years of data. A different window gives different numbers.
* **No rebalancing, dividends or currency effects** beyond what adjusted prices capture.

# Part B: Option pricing

## 5. Pricing a European call

A **European call** pays $\max(S_T - K, 0)$ at expiry $T$. **Risk-neutral pricing** says its value today is the discounted expected payoff under the risk-neutral measure $\mathbb{Q}$:

$$C_0 = e^{-rT}\,\mathbb{E}^{\mathbb{Q}}\big[\max(S_T - K, 0)\big].$$

Under **Geometric Brownian Motion**, $dS_t = r S_t\,dt + \sigma S_t\,dW_t$. Applying **Itô's lemma** to $\ln S_t$ gives an exact solution:

$$S_T = S_0 \exp\Big(\big(r - \tfrac12\sigma^2\big)T + \sigma\sqrt{T}\,Z\Big), \qquad Z \sim N(0,1).$$

A European payoff depends only on $S_T$, so **one step is exact**. Time-stepping only matters for path-dependent options. The same model also has a closed-form answer, the **Black-Scholes formula**, which gives us a true value to test the simulation against.

In [ ]:
# Contract and market inputs (option expiring 17 Mar 2022, valued on 17 Jan 2022)
S0, K = 101.15, 98.01
sigma = 0.0991          # volatility (9.91% a year)
r = 0.01                # risk-free rate (1% a year)
T = 60 / 365            # time to expiry in years
MARKET_PRICE = 3.86

def black_scholes_call(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

bs_price = black_scholes_call(S0, K, T, r, sigma)
implied_vol = brentq(lambda s: black_scholes_call(S0, K, T, r, s) - MARKET_PRICE, 1e-4, 2.0)
print(f"Black-Scholes price : {bs_price:.4f}")
print(f"Market price        : {MARKET_PRICE:.4f}")
print(f"Implied volatility  : {implied_vol:.2%}  (vs 9.91% input): the market is pricing in more uncertainty")

## 5.1 Plain Monte Carlo

In [ ]:
def terminal_prices(Z):
    return S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)

def summarise(samples, label, seconds):
    est = samples.mean(); se = samples.std(ddof=1) / np.sqrt(len(samples))
    return {"Method": label, "Price": est, "Std error": se,
            "95% CI low": est - 1.96 * se, "95% CI high": est + 1.96 * se,
            "Error vs BS": est - bs_price, "Time (ms)": 1000 * seconds}

M = 100_000
disc = np.exp(-r * T)

t0 = time.perf_counter()
Z = np.random.default_rng(SEED).standard_normal(M)
plain = disc * np.maximum(terminal_prices(Z) - K, 0)
res_plain = summarise(plain, "Plain MC", time.perf_counter() - t0)
pd.Series(res_plain).drop("Method").astype(float).round(4)

## 6. Variance reduction: same accuracy for less work

### 6.1 Antithetic variates
If $Z$ is a standard normal, so is $-Z$. Pair each path with its **mirror image** and average the two payoffs. When one path is too high, its partner tends to be too low, so the errors partly cancel. We use $M/2$ pairs to keep the total number of paths equal, and compute the standard error **over pairs**, since the two halves of a pair aren't independent.

### 6.2 Control variates
Find a quantity that moves with the payoff and whose expectation we **know exactly**. Here we use the discounted terminal price $Y = e^{-rT}S_T$, with $\mathbb{E}^{\mathbb{Q}}[Y] = S_0$. We then correct each sample:

$$X^{cv} = X - \beta\,(Y - S_0), \qquad \beta = \frac{\text{Cov}(X, Y)}{\text{Var}(Y)}.$$

The correction removes the part of the noise in $X$ that is explained by $Y$. The variance falls by a factor of $1-\rho^2$, where $\rho$ is the correlation between $X$ and $Y$.

In [ ]:
# Antithetic: M/2 pairs -> M paths in total
t0 = time.perf_counter()
Zh = np.random.default_rng(SEED).standard_normal(M // 2)
pay_up = disc * np.maximum(terminal_prices(Zh) - K, 0)
pay_dn = disc * np.maximum(terminal_prices(-Zh) - K, 0)
pairs = 0.5 * (pay_up + pay_dn)
res_anti = summarise(pairs, "Antithetic variates", time.perf_counter() - t0)

# Control variate on the discounted terminal price
t0 = time.perf_counter()
Z = np.random.default_rng(SEED).standard_normal(M)
ST = terminal_prices(Z)
X = disc * np.maximum(ST - K, 0)
Y = disc * ST
beta = np.cov(X, Y, ddof=1)[0, 1] / Y.var(ddof=1)
cv = X - beta * (Y - S0)
res_cv = summarise(cv, "Control variate", time.perf_counter() - t0)
rho = np.corrcoef(X, Y)[0, 1]
print(f"beta = {beta:.3f}, corr(X, Y) = {rho:.3f} -> theoretical variance cut 1 - rho^2 = {1 - rho**2:.3f}")

results = pd.DataFrame([res_plain, res_anti, res_cv]).set_index("Method")
results["Variance reduction (x)"] = (res_plain["Std error"] / results["Std error"]) ** 2
results.loc["Black-Scholes (exact)"] = [bs_price, 0, bs_price, bs_price, 0, np.nan, np.nan]
display(results.round(4))

**How to read the table.** A variance reduction of *k*x means the method reaches the same standard error as plain Monte Carlo with roughly **k times fewer simulations**. Every 95% interval should contain the Black-Scholes price. If it didn't, that would point to a bug.

## 7. Convergence: watching the $1/\sqrt{M}$ law

We re-price with growing numbers of simulations and plot the estimate with its 95% band against the exact Black-Scholes value.

In [ ]:
Ms = np.unique(np.logspace(2, 6, 25).astype(int))
g = np.random.default_rng(SEED)
conv = []
for m in Ms:
    Z = g.standard_normal(m)
    ST = terminal_prices(Z)
    X = disc * np.maximum(ST - K, 0)
    Y = disc * ST
    b = np.cov(X, Y, ddof=1)[0, 1] / Y.var(ddof=1)
    for label, s in (("Plain MC", X), ("Control variate", X - b * (Y - S0))):
        conv.append({"M": m, "method": label, "price": s.mean(), "se": s.std(ddof=1) / np.sqrt(m)})
conv = pd.DataFrame(conv)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for label, color in (("Plain MC", "tab:blue"), ("Control variate", "tab:green")):
    d = conv[conv.method == label]
    axes[0].plot(d.M, d.price, color=color, label=label)
    axes[0].fill_between(d.M, d.price - 1.96 * d.se, d.price + 1.96 * d.se, color=color, alpha=0.15)
    axes[1].loglog(d.M, d.se, "o-", color=color, ms=3, label=label)
axes[0].axhline(bs_price, color="k", ls="--", label=f"Black-Scholes {bs_price:.3f}")
axes[0].set_xscale("log"); axes[0].set_xlabel("Number of simulations M"); axes[0].set_ylabel("Price")
axes[0].set_title("Estimate and 95% confidence band"); axes[0].legend()
ref = conv[conv.method == "Plain MC"].se.iloc[0] * np.sqrt(Ms[0] / Ms)
axes[1].loglog(Ms, ref, "k:", label=r"$\propto 1/\sqrt{M}$")
axes[1].set_xlabel("Number of simulations M"); axes[1].set_ylabel("Standard error")
axes[1].set_title("Standard error falls like 1/sqrt(M)"); axes[1].legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "convergence.png", dpi=150); plt.show()

## 8. Results summary

The next cell prints the headline numbers in one place, so you can paste them into the README and your CV.

In [ ]:
v95, e95 = var_es(final, V0, 0.95)
v995, e995 = var_es(final, V0, 0.995)
print(f"Data source: {DATA_SOURCE}")
print(f"PORTFOLIO (equal-weighted, 6 European insurers, €{V0:,.0f}, 1 year, {N_PATHS:,} paths)")
print(f"  95% VaR   €{v95:,.0f}   | 95% ES   €{e95:,.0f}")
print(f"  99.5% VaR €{v995:,.0f}   | 99.5% ES €{e995:,.0f}")
print(f"  Ignoring correlation understates 99.5% VaR by {100 * (1 - v_ind / v_corr):.0f}%")
print(f"OPTION (European call, M = {M:,})")
print(f"  Black-Scholes {bs_price:.4f} | market {MARKET_PRICE} | implied vol {implied_vol:.2%}")
for name, row in results.drop('Black-Scholes (exact)').iterrows():
    print(f"  {name:<20} {row['Price']:.4f} ± {row['Std error']:.4f}  (variance reduction {row['Variance reduction (x)']:.1f}x)")

## 9. Conclusions and next steps

* Monte Carlo reproduces the Black-Scholes price within its confidence interval. That's the key check that the simulation engine is correct.
* Control variates gave the biggest efficiency gain because the payoff is highly correlated with the terminal price. Antithetic variates help less, since the call payoff is not a linear function of $Z$.
* For the insurer portfolio, the 99.5% one-year VaR (the Solvency II calibration level) is far larger than the 95% figure, and ignoring correlation would materially understate it.

**Next steps:** fat-tailed returns (Student-t), GARCH volatility, backtesting VaR against realised losses, quasi-random (Sobol) sequences, and pricing path-dependent options such as Asian options, where Monte Carlo has no closed-form alternative.